# production health paper execution v1

验证 production health 和 dashboard 可以读取 production manifest 中的 paper execution schedule 管线状态。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dashboard import build_dashboard_view_model, write_dashboard_html, write_model_matrix_dashboard_export
from src.schedule import ProductionHealthCheckConfig, run_production_health_check_once

CARD = Path('research/production_health_paper_execution_v1')
CARD.mkdir(parents=True, exist_ok=True)
HEALTH_PATH = CARD / 'production_health_export.json'
DASHBOARD_EXPORT_PATH = CARD / 'dashboard_export.json'
DASHBOARD_HTML_PATH = CARD / 'dashboard.html'
SUMMARY_PATH = CARD / 'analysis_summary.json'

In [2]:
events: list[str] = []
health_config = ProductionHealthCheckConfig(
    name='production_health_paper_execution_v1',
    registry_path=Path('production/production_registry.json'),
    strategy_id='paper_sim_replay_v1',
    output_path=HEALTH_PATH,
    notify_details=True,
)
health_result = run_production_health_check_once(
    health_config,
    lock_dir=CARD / 'locks',
    notify=events.append,
)
health_payload = json.loads(HEALTH_PATH.read_text(encoding='utf-8'))

assert health_result.status == 'success'
assert health_payload['schema_version'] == 'schedule.production_health.v1'
assert health_payload['health_status'] == 'healthy'
assert health_payload['paper_execution_pipeline_schema_version'] == 'schedule.paper_execution.v1'
assert health_payload['paper_execution_job_status'] == 'success'

health_payload

{'schema_version': 'schedule.production_health.v1',
 'strategy_id': 'paper_sim_replay_v1',
 'health_status': 'healthy',
 'is_healthy': True,
 'registry_path': 'production/production_registry.json',
 'manifest_path': 'production/paper_sim_replay_v1/production_manifest.json',
 'manifest_hash': '442f9f89782ff2c8ae0711220ca3486cbb32e07a4233d2c31a8b4b9558992c25',
 'registry_manifest_hash': '442f9f89782ff2c8ae0711220ca3486cbb32e07a4233d2c31a8b4b9558992c25',
 'computed_manifest_hash': '442f9f89782ff2c8ae0711220ca3486cbb32e07a4233d2c31a8b4b9558992c25',
 'manifest_hash_matches_registry': True,
 'manifest_hash_matches_content': True,
 'artifact_id': 'paper_sim_replay_v1',
 'artifact_type': 'production_strategy',
 'mode': 'paper',
 'status': 'paper_replay_completed',
 'live_broker_enabled': False,
 'paper_execution_pipeline_schema_version': 'schedule.paper_execution.v1',
 'paper_execution_job_status': 'success',
 'published_at': '2026-06-09T21:30:00Z',
 'start': '2026-01-02',
 'end': '2026-06-09'

In [3]:
dashboard_payload = write_model_matrix_dashboard_export(
    'production/paper_sim_replay_v1/matrix_card',
    output_path=DASHBOARD_EXPORT_PATH,
    top_n=3,
    production_health_path=HEALTH_PATH,
)
view_model = build_dashboard_view_model(dashboard_payload)
html = write_dashboard_html(view_model, output_path=DASHBOARD_HTML_PATH, title='SOXL Production Health')

assert dashboard_payload['production_health']['paper_execution_pipeline_schema_version'] == 'schedule.paper_execution.v1'
assert view_model['production_health']['paper_execution_job_status'] == 'success'
assert 'Paper Execution' in html
assert 'schedule.paper_execution.v1' in html

{
    'dashboard_schema_version': dashboard_payload['schema_version'],
    'dashboard_artifact_id': dashboard_payload['artifact_id'],
    'view_model_schema_version': view_model['schema_version'],
    'html_size_bytes': len(html.encode('utf-8')),
}

{'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'dashboard_artifact_id': 'matrix_card',
 'view_model_schema_version': 'dashboard.view_model.v1',
 'html_size_bytes': 358009}

In [4]:
summary = {
    'schema_version': 'research.production_health_paper_execution_v1.summary',
    'status': 'completed',
    'strategy_id': health_payload['strategy_id'],
    'health_status': health_payload['health_status'],
    'is_healthy': health_payload['is_healthy'],
    'paper_execution_pipeline_schema_version': health_payload['paper_execution_pipeline_schema_version'],
    'paper_execution_job_status': health_payload['paper_execution_job_status'],
    'live_broker_enabled': health_payload['live_broker_enabled'],
    'manifest_hash': health_payload['manifest_hash'],
    'dashboard_export': str(DASHBOARD_EXPORT_PATH),
    'dashboard_html': str(DASHBOARD_HTML_PATH),
    'production_health_export': str(HEALTH_PATH),
    'notifications': events,
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

summary

{'schema_version': 'research.production_health_paper_execution_v1.summary',
 'status': 'completed',
 'strategy_id': 'paper_sim_replay_v1',
 'health_status': 'healthy',
 'is_healthy': True,
 'paper_execution_pipeline_schema_version': 'schedule.paper_execution.v1',
 'paper_execution_job_status': 'success',
 'live_broker_enabled': False,
 'manifest_hash': '442f9f89782ff2c8ae0711220ca3486cbb32e07a4233d2c31a8b4b9558992c25',
 'dashboard_export': 'research/production_health_paper_execution_v1/dashboard_export.json',
 'dashboard_html': 'research/production_health_paper_execution_v1/dashboard.html',
 'production_health_export': 'research/production_health_paper_execution_v1/production_health_export.json',
 'notifications': ['production_health_paper_execution_v1:success | strategy=paper_sim_replay_v1 | health=healthy | missing=0 | paper_execution=schedule.paper_execution.v1/success | manifest=production/paper_sim_replay_v1/production_manifest.json']}

## 中文结论

- 做了什么：用 `src.schedule.run_production_health_check_once` 读取 `production/production_registry.json`，把 `paper_sim_replay_v1` 的 production manifest 健康状态导出到 `production_health_export.json`，再通过 `src.dashboard.write_model_matrix_dashboard_export` 和 `write_dashboard_html` 接入 dashboard。
- 核心指标：`health_status=healthy`、`paper_execution_pipeline_schema_version=schedule.paper_execution.v1`、`paper_execution_job_status=success`、`live_broker_enabled=False`，HTML 中可见 `Paper Execution` 面板字段。
- regime 差异：本实验不重新训练或回测 regime，只验证 production health 对已生成的 analysis/dashboard 产物增加 paper execution 管线可见性。
- 是否晋升：可作为 dashboard/health 链路证据保留；不是新策略晋升，也不启用 live broker。
- 风险：health 只证明 manifest、sidecar 路径、content hash 和 paper execution 记录一致，不证明真实 broker 可用；真实 broker 仍在 v1 范围外。
- 下一步：后续 schedule/dashboard 刷新可直接消费该 health export，把 paper execution 状态纳入日常页面和通知。